In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/ml-100k")

ratings = pd.read_csv(
    DATA_PATH / "u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    DATA_PATH / "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["movie_id", "title"]
)

train, test = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

In [2]:
user_ids = sorted(ratings["user_id"].unique())
movie_ids = sorted(ratings["movie_id"].unique())

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_ids)
}

num_users = len(user_ids)
num_movies = len(movie_ids)

In [3]:
import sys

sys.path.append("../src")

from matrix_factorization import train_matrix_factorization

In [4]:
P, Q, user_bias, movie_bias, global_mean, training_rmse = (
    train_matrix_factorization(
        train=train,
        user_to_idx=user_to_idx,
        movie_to_idx=movie_to_idx,
        num_users=num_users,
        num_movies=num_movies,
        n_factors=10,
        learning_rate=0.005,
        regularization=0.02,
        epochs=20,
        seed=42
    )
)

Epoch  1/20 - Training RMSE: 1.0486
Epoch  2/20 - Training RMSE: 0.9836
Epoch  3/20 - Training RMSE: 0.9611
Epoch  4/20 - Training RMSE: 0.9489
Epoch  5/20 - Training RMSE: 0.9408
Epoch  6/20 - Training RMSE: 0.9349
Epoch  7/20 - Training RMSE: 0.9302
Epoch  8/20 - Training RMSE: 0.9263
Epoch  9/20 - Training RMSE: 0.9230
Epoch 10/20 - Training RMSE: 0.9200
Epoch 11/20 - Training RMSE: 0.9171
Epoch 12/20 - Training RMSE: 0.9143
Epoch 13/20 - Training RMSE: 0.9114
Epoch 14/20 - Training RMSE: 0.9085
Epoch 15/20 - Training RMSE: 0.9053
Epoch 16/20 - Training RMSE: 0.9020
Epoch 17/20 - Training RMSE: 0.8982
Epoch 18/20 - Training RMSE: 0.8941
Epoch 19/20 - Training RMSE: 0.8896
Epoch 20/20 - Training RMSE: 0.8849


In [6]:
def predict_rating(user_id, movie_id):
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        return global_mean

    u = user_to_idx[user_id]
    i = movie_to_idx[movie_id]

    prediction = (
        global_mean
        + user_bias[u]
        + movie_bias[i]
        + np.dot(P[u], Q[i])
    )

    return np.clip(prediction, 1, 5)

In [7]:
def recommend_movies(user_id, n=10):
    if user_id not in user_to_idx:
        raise ValueError(f"User {user_id} not found.")

    rated_movies = set(
        ratings.loc[
            ratings["user_id"] == user_id,
            "movie_id"
        ]
    )

    recommendations = []

    for movie_id in movie_ids:
        if movie_id not in rated_movies:
            predicted_rating = predict_rating(
                user_id,
                movie_id
            )

            recommendations.append(
                (movie_id, predicted_rating)
            )

    recommendations.sort(
        key=lambda x: x[1],
        reverse=True
    )

    top_recommendations = recommendations[:n]

    result = pd.DataFrame(
        top_recommendations,
        columns=["movie_id", "predicted_rating"]
    )

    result = result.merge(
        movies,
        on="movie_id"
    )

    return result[
        ["movie_id", "title", "predicted_rating"]
    ]

In [8]:
recommend_movies(1, n=10)

,movie_id,title,predicted_rating
0,318,Schindler's List (1993),4.607271
1,408,"Close Shave, A (1995)",4.593301
2,285,Secrets & Lies (1996),4.529625
3,480,North by Northwest (1959),4.495565
4,483,Casablanca (1942),4.465402
5,357,One Flew Over the Cuckoo's Nest (1975),4.444291
6,479,Vertigo (1958),4.433831
7,657,"Manchurian Candidate, The (1962)",4.411345
8,474,Dr. Strangelove or: How I Learned to Stop Worr...,4.395165
9,511,Lawrence of Arabia (1962),4.393061


In [9]:
def favorite_movies(user_id, n=10):
    user_ratings = ratings[
        ratings["user_id"] == user_id
    ]

    favorites = (
        user_ratings
        .merge(movies, on="movie_id")
        .sort_values(
            ["rating", "movie_id"],
            ascending=[False, True]
        )
        .head(n)
    )

    return favorites[
        ["movie_id", "title", "rating"]
    ]

In [10]:
favorite_movies(1, n=10)

,movie_id,title,rating
174,1,Toy Story (1995),5
50,6,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,5
138,9,Dead Man Walking (1995),5
74,12,"Usual Suspects, The (1995)",5
266,13,Mighty Aphrodite (1995),5
75,14,"Postino, Il (1994)",5
262,15,Mr. Holland's Opus (1995),5
219,16,French Twist (Gazon maudit) (1995),5
258,19,Antonia's Line (1995),5
157,32,Crumb (1994),5


## Personalized Recommendations

The trained matrix factorization model can generate personalized recommendations by predicting a rating for every movie a user has not previously rated and ranking those movies by predicted preference.

For User 1, the model recommended highly rated films including *Schindler's List*, *A Close Shave*, *North by Northwest*, *Casablanca*, and *One Flew Over the Cuckoo's Nest*.

The recommendations demonstrate how the learned user and movie latent factors can be used for more than rating prediction. However, qualitative inspection alone cannot determine recommendation quality. The final evaluation will therefore use ranking-based metrics to measure whether highly ranked recommendations correspond to movies users actually liked.